# trace viz

Tour of the visualization helpers that ship with rlmflow, anchored to the deterministic needle-in-haystack trace under `docs/generated/needle_trace/`. Regenerate with `python examples/blog_needle_graph.py` if you change the engine.

Everything below operates on a `Graph` (single snapshot) or `list[Graph]` (every step). The same renders are available via the CLI as `rlmflow render <path> -f F`.

In [ ]:
from pathlib import Path

from rlmflow.utils.trace import load_trace

TRACE_PATH = Path("./generated/needle_trace/")

trace = load_trace(TRACE_PATH)
graphs = trace.graphs
final = graphs[-1]

print(f"loaded {len(graphs)} snapshots from {TRACE_PATH}")
print(f"metadata: {trace.metadata}")
cur = final.current()
print(f"final: {final.root_agent_id} [{cur.type if cur else 'empty'}] -> {final.result()[:80]}")

## Transcript

`graph.transcript()` renders the root agent's chat-log; pass an agent id to focus on any child.

In [ ]:
print(final.transcript(include_system=False))

## Plotly graph

`graph.plot()` is the same Plotly figure the viewer renders — every node laid out as a tidy tree, colored by type. Hover for token / depth / model annotations.

In [ ]:
graphs[2].plot()

## Interactive viewer

`open_viewer(graphs)` boots an inline Gradio stepper: step slider, clickable nodes, per-state detail.

In [ ]:
from rlmflow.utils.viewer import open_viewer

open_viewer(graphs, inline=True, height=720, quiet=True)

## Step explorer

Pair `graph.tree()` with `graph.transcript()` for a step-by-step readout.

In [ ]:
def show_step(i: int, *, transcript: bool = True):
    """Render one trace snapshot by index."""
    graph = graphs[i]
    cur = graph.current()
    print(
        f"step {i} / {len(graphs) - 1}: {graph.root_agent_id} "
        f"[{cur.type if cur else 'empty'}]"
    )
    print(graph.tree())
    if transcript:
        print("\n--- transcript ---")
        print(graph.transcript(include_system=False))
    return graph.plot(title=f"step {i} / {len(graphs) - 1}")


show_step(0, transcript=False)

## Static exports

`save_steps(graphs, out_dir)` writes one PNG per snapshot. Requires `kaleido` for the image backend (`pip install kaleido`).

In [ ]:
from rlmflow.utils.viewer import save_steps

save_steps(
    graphs,
    "./static/needle_trace_images",
    fmt="png",
    width=1800,
    height=1350,
    scale=2.0,
    element_mult=1.0,
    margin={"l": 24, "r": 24, "t": 80, "b": 120},
)

## ipywidgets slider

Drop a slider on top of `show_step` for the easiest live exploration.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    slider = widgets.IntSlider(
        value=0,
        min=0,
        max=len(graphs) - 1,
        step=1,
        description="step",
        continuous_update=False,
    )
    transcript_toggle = widgets.Checkbox(value=False, description="transcript")

    out = widgets.interactive_output(
        lambda i, transcript: display(show_step(i, transcript=transcript)),
        {"i": slider, "transcript": transcript_toggle},
    )
    display(widgets.VBox([widgets.HBox([slider, transcript_toggle]), out]))
except ImportError:
    print("ipywidgets is not installed. Use show_step(i) manually instead.")

## Other views from the same snapshots

In [ ]:
mermaid_flowchart = final.plot("flowchart")
sequence_diagram = final.plot("sequence")
gantt_html = final.plot("gantt")

print(mermaid_flowchart[:500])